# Diagnóstico da API IFGProduz

Este notebook testa a integração com a API do IFGProduz.

**Pré-requisito:** coloque a URL do seu Lattes na célula de configuração abaixo.

## 1. Configuração

In [ ]:
import urllib.request, urllib.parse, json

# ← EDITE AQUI
LATTES_URL = "http://lattes.cnpq.br/0000000000000000"

BASE_API = "https://api.lattes.bcc.ifg.edu.br/api/informacoes_docentes"

lattes_id = urllib.parse.quote(LATTES_URL.strip(), safe=":/")
URL = f"{BASE_API}?infor_docentes=informacoes_docentesProducao&lattes_id={lattes_id}"
print("URL que será usada:", URL)

## 2. Buscar resposta da API

In [ ]:
req = urllib.request.Request(URL, headers={"Accept": "application/json"})
with urllib.request.urlopen(req, timeout=10) as r:
    data = json.loads(r.read().decode())

print("Chaves na raiz da resposta:", list(data.keys()))

## 3. Ver todos os itens de dados_pdf

In [ ]:
dados = data["dados_pdf"]

print(f"{'Item':<45} {'Quantidade':>10} {'Peso':>8} {'Subtotal':>10}")
print("-" * 78)

total = 0.0
for k, v in dados.items():
    if isinstance(v, (int, float)):
        subtotal = float(v)
        print(f"{k:<45} {'(fixo)':>10} {'':>8} {subtotal:>10.2f}")
        total += subtotal
    elif isinstance(v, dict) and "quantidade" in v and "peso" in v:
        qtd = v["quantidade"]
        peso = v["peso"]
        subtotal = qtd * peso
        if qtd > 0:
            print(f"{k:<45} {qtd:>10} {peso:>8} {subtotal:>10.2f}")
        total += subtotal

print("-" * 78)
print(f"{'TOTAL CALCULADO':<45} {'':>10} {'':>8} {total:>10.2f}")
print(f"\nValor que o sistema vai mostrar (máx 100): {min(total, 100):.2f}")

## 4. Simular a função corrigida do sistema

In [ ]:
def buscar_pontuacao_ifgproduz(lattes_url):
    if not lattes_url or not lattes_url.strip():
        return None
    lattes_id = urllib.parse.quote(lattes_url.strip(), safe=":/")
    url = (
        "https://api.lattes.bcc.ifg.edu.br/api/informacoes_docentes"
        f"?infor_docentes=informacoes_docentesProducao&lattes_id={lattes_id}"
    )
    try:
        req = urllib.request.Request(url, headers={"Accept": "application/json"})
        with urllib.request.urlopen(req, timeout=10) as response:
            dados = json.loads(response.read().decode())["dados_pdf"]
        total = 0.0
        for valor in dados.values():
            if isinstance(valor, (int, float)):
                total += valor
            elif isinstance(valor, dict) and "quantidade" in valor and "peso" in valor:
                total += valor["quantidade"] * valor["peso"]
        return total
    except Exception as e:
        print(f"Erro: {type(e).__name__}: {e}")
        return None

resultado = buscar_pontuacao_ifgproduz(LATTES_URL)
print(f"Resultado bruto : {resultado}")
if resultado is not None:
    print(f"Valor no sistema: {min(resultado, 100):.1f} pts (máx 100)")